In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
try:
#     plt.style.use('belle2')
    plt.style.use('belle2_serif')
#     plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter


Welcome to JupyROOT 6.26/04


In [2]:
from math import sqrt

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    return central_value, error

In [3]:
def combine_x_plus_y_divided_by_2(x, y, x_err, y_err):
    central_value = (x+y)/2
    error = sqrt(x_err**2 +  y_err**2)/2
    return central_value, error

In [4]:
def correct_Acp_stats( Araw, Araw_err, Aref, Aref_err, Aref_pdg, Aref_K_mix):
    final_Acp = Araw - Aref + Aref_pdg + Aref_K_mix
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)

    return final_Acp, final_Acp_err

In [5]:
def correct_Acp_stats_no_Kmix( Araw, Araw_err, Aref, Aref_err, Aref_pdg):
    final_Acp = Araw - Aref + Aref_pdg 
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)

    return final_Acp, final_Acp_err

In [6]:
def delta_Acp_sys_unc(A_original, A_original_error, A, A_error):
    delta_Acp = A - A_original
    if A_original_error > A_error:
        delta_Acp_error = sqrt(A_original_error**2 - A_error**2)
    elif A_original_error < A_error:
        delta_Acp_error = sqrt(A_error**2 - A_original_error**2)
    else: 
        print("Error: unable to proceed.")
        sys.exit()

    # print(f"delta_Acp: {delta_Acp}, delta_Acp_error: {delta_Acp_error}")
    print(f"Original Acp: {A_original * 100:.5f}%, Original Acp error: {A_original_error * 100:.5f}%")
    print(f"Acp: {A * 100:.5f}%, Acp error: {A_error * 100:.5f}%")
    print(f"delta_Acp: {delta_Acp * 100:.5f}%, delta_Acp_error: {delta_Acp_error * 100:.5f}%")
    return delta_Acp, delta_Acp_error

# proc13

## Acp(D+ -> eta pi+)

### eta -> gg

In [90]:
#fit_v12
Araw_gg_cms_plus = -0.012178669499235406
Araw_gg_cms_plus_error =  0.00833531629916151
Araw_gg_cms_minus = 0.02006047616581541
Araw_gg_cms_minus_error =  0.009524993080968436
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.39409%, Araw_gg_stats_error: 0.63286%


In [91]:
#fit_v3
Aref_gg_cms_plus = -0.018238433045352664
Aref_gg_cms_plus_error = 0.002224581669675485
Aref_gg_cms_minus = 0.021783531783850174
Aref_gg_cms_minus_error = 0.0024979544820659116
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.17725%, Aref_gg_stats_error: 0.16725%


In [92]:
Aref_gg_pdg = 0
Acp_etapip_gg_value, Acp_etapip_gg_error = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: 0.21684%, Acp_etapip_gg_error: 0.65458%


In [93]:
AK0bar = - 0.004227
AK0bar_err = 0.00001
Acp_etapip_gg_value += AK0bar
Acp_etapip_gg_error = math.sqrt(Acp_etapip_gg_error**2 + AK0bar_err**2)
print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: -0.20586%, Acp_etapip_gg_error: 0.65458%


### eta -> pipipi

In [94]:
#fitv12
Araw_3pi_cms_plus = -0.021402010529292026
Araw_3pi_cms_plus_error = 0.010971959712653578
Araw_3pi_cms_minus = 0.03799187984462482
Araw_3pi_cms_minus_error =  0.012190270823573046

Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.82949%, Araw_3pi_stats_error: 0.82004%


In [95]:
#fitv3
Aref_3pi_cms_plus = -0.018228713603259727
Aref_3pi_cms_plus_error =   0.0020528872348583582
Aref_3pi_cms_minus = 0.023383757265216243
Aref_3pi_cms_minus_error =   0.002312931060067025
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.25775%, Aref_pipipi_stats_error: 0.15463%


In [96]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value, Acp_etapip_pipipi_error =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: 0.57174%, Acp_etapip_pipipi_error: 0.83449%


In [97]:
AK0bar = - 0.004213
AK0bar_err = 0.000009
Acp_etapip_pipipi_value += AK0bar
Acp_etapip_pipipi_error = math.sqrt(Acp_etapip_pipipi_error ** 2 + AK0bar_err**2)
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: 0.15044%, Acp_etapip_pipipi_error: 0.83449%


In [98]:
combined_central_value, combined_error = combine_error_weighted(Acp_etapip_gg_value, Acp_etapip_pipipi_value, Acp_etapip_gg_error, Acp_etapip_pipipi_error)

print(f"Acp_combined: {combined_central_value * 100:.5f}%, Acp_combined_error: {combined_error * 100:.5f}%")

Acp_combined: -0.07014%, Acp_combined_error: 0.51504%


## Acp(Ds+ -> eta pi+)

### eta -> gg

In [99]:
#fit_v12
Araw_gg_cms_plus = -0.015366934811480326
Araw_gg_cms_plus_error = 0.005950863963835207
Araw_gg_cms_minus = 0.02476444396520372
Araw_gg_cms_minus_error = 0.0067583873350745005
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 0.46988%, Araw_gg_stats_error: 0.45025%


In [100]:
#fit_v3
Aref_gg_cms_plus = -0.018238433045352664
Aref_gg_cms_plus_error = 0.002224581669675485
Aref_gg_cms_minus = 0.021783531783850174
Aref_gg_cms_minus_error = 0.0024979544820659116
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.17725%, Aref_gg_stats_error: 0.16725%


In [101]:
Aref_gg_pdg = 0
Acp_etapip_gg_value, Acp_etapip_gg_error = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: 0.29262%, Acp_etapip_gg_error: 0.48030%


In [102]:
AK0bar = - 0.004227
AK0bar_err = 0.00001
Acp_etapip_gg_value += AK0bar
Acp_etapip_gg_error = math.sqrt(Acp_etapip_gg_error**2 + AK0bar_err**2)
print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: -0.13008%, Acp_etapip_gg_error: 0.48031%


### eta -> pipipi

In [103]:
#fitv12
Araw_3pi_cms_plus = -0.01639963140668954
Araw_3pi_cms_plus_error = 0.008080155150900381
Araw_3pi_cms_minus = 0.017065419785479152
Araw_3pi_cms_minus_error =  0.008906084361279487
Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.03329%, Araw_3pi_stats_error: 0.60126%


In [104]:
#fitv3
Aref_3pi_cms_plus = -0.018228713603259727
Aref_3pi_cms_plus_error =   0.0020528872348583582
Aref_3pi_cms_minus = 0.023383757265216243
Aref_3pi_cms_minus_error =   0.002312931060067025
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.25775%, Aref_pipipi_stats_error: 0.15463%


In [105]:
# -0.22446 - 0.421

In [106]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value, Acp_etapip_pipipi_error =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: -0.22446%, Acp_etapip_pipipi_error: 0.62083%


In [107]:
AK0bar = - 0.004213
AK0bar_err = 0.000009
Acp_etapip_pipipi_value += AK0bar
Acp_etapip_pipipi_error = math.sqrt(Acp_etapip_pipipi_error ** 2 + AK0bar_err**2)
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: -0.64576%, Acp_etapip_pipipi_error: 0.62083%


In [108]:
combined_central_value, combined_error = combine_error_weighted(Acp_etapip_gg_value, Acp_etapip_pipipi_value, Acp_etapip_gg_error, Acp_etapip_pipipi_error)

print(f"Acp_combined: {combined_central_value * 100:.5f}%, Acp_combined_error: {combined_error * 100:.5f}%")

Acp_combined: -0.32317%, Acp_combined_error: 0.37989%


## Acp(D+ -> eta K+)

### eta -> gg

In [109]:
#fitv15
Araw_gg_cms_plus = -0.1490072757648787
Araw_gg_cms_plus_error = 0.14162171540540058
Araw_gg_cms_minus = -0.004733925765426861
Araw_gg_cms_minus_error = 0.11057573900735723
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: -7.68706%, Araw_gg_stats_error: 8.98383%


In [110]:
#fitv3
Aref_gg_cms_plus = -0.020433612167617854
Aref_gg_cms_plus_error = 0.005354808253089316
Aref_gg_cms_minus = 0.0191431504521844
Aref_gg_cms_minus_error = 0.00565186380693572
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: -0.06452%, Aref_gg_stats_error: 0.38929%


In [111]:
# -7.67366 - 0.428

In [112]:
Aref_gg_pdg = 0
Acp_etapip_gg_value, Acp_etapip_gg_error = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: -7.62254%, Acp_etapip_gg_error: 8.99226%


In [113]:
AK0bar = - 0.004279
AK0bar_err = 0.000028
Acp_etapip_gg_value += AK0bar
Acp_etapip_gg_error = math.sqrt(Acp_etapip_gg_error**2 + AK0bar_err**2)
print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: -8.05044%, Acp_etapip_gg_error: 8.99226%


### eta -> pipipi

In [114]:
#fitv12
Araw_3pi_cms_plus = 0.11929404986386816
Araw_3pi_cms_plus_error = 0.13422475613672724
Araw_3pi_cms_minus = 0.0003133040074678739
Araw_3pi_cms_minus_error = 0.13250925422441795
Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 5.98037%, Araw_3pi_stats_error: 9.43067%


In [115]:
#fitv3
Aref_3pi_cms_plus = -0.020834298244178706
Aref_3pi_cms_plus_error =  0.004534553386959683
Aref_3pi_cms_minus = 0.02314459586172979
Aref_3pi_cms_minus_error =  0.00482307642802815
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.11551%, Aref_pipipi_stats_error: 0.33100%


In [116]:
# 5.97831 - 0.424

In [117]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value, Acp_etapip_pipipi_error =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: 5.86485%, Acp_etapip_pipipi_error: 9.43647%


In [118]:
AK0bar = - 0.004237
AK0bar_err = 0.000021
Acp_etapip_pipipi_value += AK0bar
Acp_etapip_pipipi_error = math.sqrt(Acp_etapip_pipipi_error ** 2 + AK0bar_err**2)
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: 5.44115%, Acp_etapip_pipipi_error: 9.43647%


In [119]:
combined_central_value, combined_error = combine_error_weighted(Acp_etapip_gg_value, Acp_etapip_pipipi_value, Acp_etapip_gg_error, Acp_etapip_pipipi_error)

print(f"Acp_combined: {combined_central_value * 100:.5f}%, Acp_combined_error: {combined_error * 100:.5f}%")

Acp_combined: -1.62966%, Acp_combined_error: 6.50987%


## Acp(Ds+ -> eta K+)

### eta -> gg

In [120]:
#fitv15
Araw_gg_cms_plus = -0.023654147770188927
Araw_gg_cms_plus_error = 0.03326678161774499
Araw_gg_cms_minus = 0.023165388281711996
Araw_gg_cms_minus_error = 0.03229630301883568
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: -0.02444%, Araw_gg_stats_error: 2.31826%


In [121]:
#fitv3
Aref_gg_cms_plus = -0.020433612167617854
Aref_gg_cms_plus_error = 0.005354808253089316
Aref_gg_cms_minus = 0.0191431504521844
Aref_gg_cms_minus_error = 0.00565186380693572
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: -0.06452%, Aref_gg_stats_error: 0.38929%


In [122]:
# -0.06452 -0.428

In [123]:
Aref_gg_pdg = 0
Acp_etapip_gg_value, Acp_etapip_gg_error = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: 0.04009%, Acp_etapip_gg_error: 2.35072%


In [124]:
AK0bar = - 0.004279
AK0bar_err = 0.000028
Acp_etapip_gg_value += AK0bar
Acp_etapip_gg_error = math.sqrt(Acp_etapip_gg_error**2 + AK0bar_err**2)
print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: -0.38781%, Acp_etapip_gg_error: 2.35072%


### eta -> pipipi

In [125]:
#fitv15
Araw_3pi_cms_plus = -0.030063325305354494
Araw_3pi_cms_plus_error = 0.042474619005372694
Araw_3pi_cms_minus = 0.03402901030602723
Araw_3pi_cms_minus_error = 0.04141468452870767
Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 0.19828%, Araw_3pi_stats_error: 2.96617%


In [126]:
#fitv3
Aref_3pi_cms_plus = -0.020834298244178706
Aref_3pi_cms_plus_error =  0.004534553386959683
Aref_3pi_cms_minus = 0.02314459586172979
Aref_3pi_cms_minus_error =  0.00482307642802815
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.11551%, Aref_pipipi_stats_error: 0.33100%


In [127]:
# 0.07225 - 0.424

In [128]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value, Acp_etapip_pipipi_error =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: 0.08277%, Acp_etapip_pipipi_error: 2.98458%


In [129]:
AK0bar = - 0.004237
AK0bar_err = 0.000021
Acp_etapip_pipipi_value += AK0bar
Acp_etapip_pipipi_error = math.sqrt(Acp_etapip_pipipi_error ** 2 + AK0bar_err**2)
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: -0.34093%, Acp_etapip_pipipi_error: 2.98458%


In [130]:
combined_central_value, combined_error = combine_error_weighted(Acp_etapip_gg_value, Acp_etapip_pipipi_value, Acp_etapip_gg_error, Acp_etapip_pipipi_error)

print(f"Acp_combined: {combined_central_value * 100:.5f}%, Acp_combined_error: {combined_error * 100:.5f}%")

Acp_combined: -0.36987%, Acp_combined_error: 1.84670%
